# Kafka inference demonstration: historical India AQI

**Samuel Flores and Santiago Becerra · ETL G51**

This notebook coordinates the real Python producer and consumer. It uses the validated PostgreSQL warehouse and the serialized next-day model. It creates no synthetic observations.

Run it in **local Jupyter** from this repository using the same Python environment as the application. The setup in the README must already have started PostgreSQL and Kafka, populated the approved warehouse, and trained `artifacts/model.joblib` with its matching `metadata.json`.

This is not a Colab-ready Docker notebook: an ordinary Colab runtime does not provide this local Docker environment. The earlier Colab EDA/ETL notebooks remain preserved separately.


## 1. Verify paths and configuration

Set `INDIA_DATABASE_URL` in the environment of the Jupyter kernel to the host PostgreSQL connection. For the Compose stack the database port is 55432; Kafka is exposed at localhost:19092. Credentials belong in the local environment, not in notebook output or committed cells.

The consumer and producer run as separate processes. A fresh demo topic and prediction store isolate this notebook from earlier messages. The normal application topic remains `india.aqi.features.v1`.


In [1]:
from pathlib import Path
import json
import os
import sqlite3
import subprocess
import sys
import time
import uuid

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "final_stream.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "final_stream.py").exists(), "Open this notebook from the repository or its notebooks directory."
assert (ROOT / "artifacts" / "model.joblib").exists(), "Train the model first; see the README."
assert (ROOT / "artifacts" / "metadata.json").exists(), "Matching model metadata is required."
assert os.environ.get("INDIA_DATABASE_URL"), "Set INDIA_DATABASE_URL in the kernel environment first."

run_id = uuid.uuid4().hex
TOPIC = "india.aqi.features.demo." + run_id[:12]
GROUP = "india-notebook-" + run_id[:12]
BOOTSTRAP = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "localhost:19092")
STORE = ROOT / "runtime" / ("demo-" + run_id[:12] + ".sqlite")
LOG_DIR = ROOT / "runtime" / ("notebook-" + run_id[:12])
LOG_DIR.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(ROOT / "src")
ENV["INDIA_PROJECT_ROOT"] = str(ROOT)
ENV["PREDICTION_STORE"] = str(STORE)
ENV["MODEL_ARTIFACTS"] = str(ROOT / "artifacts")
print("Repository:", ROOT.name)
print("Demo topic:", TOPIC)
print("Prediction store:", STORE.name)


Repository: project
Demo topic: india.aqi.features.demo.3c11c2d5200e
Prediction store: demo-3c11c2d5200e.sqlite


## 2. Inspect the trained model and reserved test evaluation

The target is AQI on the following local day. The model was selected using 2022 validation data and fitted through 2022; the standard replay below requests 2023 records. Error metrics and baseline comparisons describe offline prediction quality. They do not become better because messages travel through Kafka.


In [2]:
metadata = json.loads((ROOT / "artifacts" / "metadata.json").read_text(encoding="utf-8"))
print("Model:", metadata["model_name"])
print("Version:", metadata["model_version"])
print("Fitted through target date:", metadata["fit_through"])
print("Features:", metadata["feature_columns"])
print(json.dumps(metadata["metrics"]["test"], indent=2))
print("Beats mean baseline:", metadata["beats_mean_baseline_on_test"])
print("Beats persistence baseline:", metadata["beats_persistence_on_test"])


Model: ridge
Version: 90b6dd4d28661a89
Fitted through target date: 2022-12-31
Features: ['city_name', 'month', 'day_of_week', 'aqi_lag1', 'aqi_mean7', 'pm25_lag1', 'pm10_lag1', 'no2_lag1', 'temperature_lag1', 'precipitation_lag1', 'wind_lag1']
{
  "selected_model": {
    "mae": 98.77327127908458,
    "rmse": 114.17293483538951,
    "r2": -0.001508590351658512
  },
  "mean_baseline": {
    "mae": 98.68034294617445,
    "rmse": 114.09303893248551,
    "r2": -0.00010741000590441807
  },
  "persistence_baseline": {
    "mae": 132.8740902872777,
    "rmse": 162.55093218445586,
    "r2": -1.030052539326431
  }
}
Beats mean baseline: False
Beats persistence baseline: True


## 3. Create the real Kafka topic and start the consumer

The topic has one partition. The consumer loads the fitted pipeline once, validates each JSON event and writes a prediction or rejection before confirming its offset. It never reads the dimensional warehouse.

This bounded demo uses 50 messages. Start the dashboard in the next section before starting the producer to observe results arriving.


In [3]:
script = str(ROOT / "src" / "final_stream.py")
subprocess.run([sys.executable, script, "create-topic", "--bootstrap", BOOTSTRAP,
                "--topic", TOPIC], cwd=ROOT, env=ENV, check=True, timeout=60)

processes = {}
log_handles = []

def launch(name, arguments):
    log_path = LOG_DIR / (name + ".log")
    handle = log_path.open("w", encoding="utf-8")
    log_handles.append(handle)
    flags = subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0
    process = subprocess.Popen(arguments, cwd=ROOT, env=ENV,
                               stdout=handle, stderr=subprocess.STDOUT,
                               creationflags=flags)
    processes[name] = process
    return process

consumer = launch("consumer", [sys.executable, script, "consume",
    "--bootstrap", BOOTSTRAP, "--topic", TOPIC, "--group", GROUP,
    "--artifacts", str(ROOT / "artifacts"), "--store", str(STORE),
    "--max-messages", "50", "--idle-timeout", "180"])
print("Consumer process started. Check its log if it exits before publication.")


Consumer process started. Check its log if it exits before publication.


## 4. Start the Streamlit dashboard

The dashboard reads this consumer's output store, not PostgreSQL. The fragment refreshes every two seconds. Port 8503 is separate from the Compose dashboard at port 8502. Open the link, leave the page visible and run the producer cell below.


In [4]:
dashboard = launch("dashboard", [sys.executable, "-m", "streamlit", "run",
    str(ROOT / "app" / "dashboard.py"), "--server.address=127.0.0.1",
    "--server.port=8503", "--server.headless=true", "--browser.gatherUsageStats=false"])
from IPython.display import display, Markdown
display(Markdown("[Open the demo dashboard](http://127.0.0.1:8503)"))


[Open the demo dashboard](http://127.0.0.1:8503)

## 5. Publish 50 approved historical feature records

The producer queries the validated fact-table view, builds the same features as training and publishes one event every 0.5 seconds. Publication time is current; origin and target dates are historical. The actual target-day AQI and its category are not included in the message.


In [5]:
producer = launch("producer", [sys.executable, script, "produce",
    "--bootstrap", BOOTSTRAP, "--topic", TOPIC,
    "--start", "2023-01-01", "--end", "2023-12-31",
    "--limit", "50", "--delay", "0.5", "--replay-id", run_id])
print("Replay is running in a separate process. Watch the dashboard counter and chart.")


Replay is running in a separate process. Watch the dashboard counter and chart.


## 6. Verify durable Kafka predictions

Run this after the replay. A successful process exit and 50 stored rows provide evidence for this bounded run. Kafka topic, partition and offset identify actual consumed records. This does not replace the quality-gate and leakage tests in the test suite.


In [6]:
assert producer.wait(timeout=60) == 0, (LOG_DIR / "producer.log").read_text(encoding="utf-8")
assert consumer.wait(timeout=60) == 0, (LOG_DIR / "consumer.log").read_text(encoding="utf-8")
assert STORE.exists(), "The consumer did not create its output store."
import pandas as pd
with sqlite3.connect(STORE) as connection:
    count = connection.execute("SELECT COUNT(*) FROM predictions WHERE replay_id=?", (run_id,)).fetchone()[0]
    rejected = connection.execute("SELECT COUNT(*) FROM rejected_events").fetchone()[0]
    results = pd.read_sql_query(
        "SELECT city_name, origin_date, target_date, predicted_aqi, model_version, "
        "topic, partition_id, kafka_offset, processed_at FROM predictions "
        "WHERE replay_id=? ORDER BY kafka_offset", connection, params=[run_id])
assert count == 50, f"Expected 50 predictions; received {count}. Inspect consumer.log."
assert rejected == 0, f"Unexpected rejections: {rejected}"
print("Confirmed:", count, "predictions from Kafka; rejected:", rejected)
display(results.head(10))
print((LOG_DIR / "producer.log").read_text(encoding="utf-8"))
print((LOG_DIR / "consumer.log").read_text(encoding="utf-8").splitlines()[-1])


Confirmed: 50 predictions from Kafka; rejected: 0


,city_name,origin_date,target_date,predicted_aqi,model_version,topic,partition_id,kafka_offset,processed_at
0,Bangalore,2022-12-31,2023-01-01,309.975090,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,0,2026-11-22T23:38:10.624386+00:00
1,Chennai,2022-12-31,2023-01-01,312.952809,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,1,2026-11-22T23:38:10.650909+00:00
2,Delhi,2022-12-31,2023-01-01,314.663502,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,2,2026-11-22T23:38:10.673347+00:00
3,Kolkata,2022-12-31,2023-01-01,313.337045,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,3,2026-11-22T23:38:10.701086+00:00
4,Mumbai,2022-12-31,2023-01-01,313.315034,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,4,2026-11-22T23:38:10.774689+00:00
5,Bangalore,2023-01-01,2023-01-02,316.797189,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,5,2026-11-22T23:38:11.275334+00:00
6,Chennai,2023-01-01,2023-01-02,321.394722,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,6,2026-11-22T23:38:11.793868+00:00
7,Delhi,2023-01-01,2023-01-02,312.238112,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,7,2026-11-22T23:38:12.272217+00:00
8,Kolkata,2023-01-01,2023-01-02,315.332944,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,8,2026-11-22T23:38:12.774175+00:00
9,Mumbai,2023-01-01,2023-01-02,319.536363,90b6dd4d28661a89,india.aqi.features.demo.3c11c2d5200e,0,9,2026-11-22T23:38:13.277133+00:00


{"sent": 50, "replay_id": "3c11c2d5200e4784b5d8e99f82d97616", "topic": "india.aqi.features.demo.3c11c2d5200e", "simulation": true, "completed_at": "2026-11-22T23:38:33.782665+00:00"}

{"predicted": 50, "duplicate": 0, "rejected": 0, "topic": "india.aqi.features.demo.3c11c2d5200e", "group": "india-notebook-3c11c2d5200e", "model_version": "90b6dd4d28661a89", "duration_seconds": 22.95, "completed_at": "2026-11-22T23:38:33.387881+00:00"}


## 7. Demonstrate idempotent redelivery

A fresh consumer group reads the same 50 retained Kafka messages into the same output store. The stored event IDs already exist, so the consumer should report duplicates while the prediction count stays 50. Messages are retained in Kafka after consumption.


In [7]:
duplicate_group = GROUP + "-redelivery"
result = subprocess.run([sys.executable, script, "consume",
    "--bootstrap", BOOTSTRAP, "--topic", TOPIC, "--group", duplicate_group,
    "--artifacts", str(ROOT / "artifacts"), "--store", str(STORE),
    "--max-messages", "50", "--idle-timeout", "30"],
    cwd=ROOT, env=ENV, check=True, capture_output=True, text=True, timeout=60)
summary = json.loads(result.stdout.strip().splitlines()[-1])
with sqlite3.connect(STORE) as connection:
    after = connection.execute("SELECT COUNT(*) FROM predictions WHERE replay_id=?", (run_id,)).fetchone()[0]
assert summary["duplicate"] == 50 and summary["predicted"] == 0, summary
assert after == 50
print(summary)
print("Prediction count remains:", after)


{'predicted': 0, 'duplicate': 50, 'rejected': 0, 'topic': 'india.aqi.features.demo.3c11c2d5200e', 'group': 'india-notebook-3c11c2d5200e-redelivery', 'model_version': '90b6dd4d28661a89', 'duration_seconds': 0.69, 'completed_at': '2026-11-22T23:38:38.751990+00:00'}
Prediction count remains: 50


## 8. Stop only the processes started by this notebook

This stops the notebook's dashboard and any still-running demo processes. Logs and prediction results remain available. It does not delete broker topics, database records or Docker volumes, and it does not stop other application services.


In [8]:
for name, process in processes.items():
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            print(name, "is still shutting down; stop it manually if needed.")
for handle in log_handles:
    handle.close()
print("Demo processes stopped. Logs:", LOG_DIR.name)


Demo processes stopped. Logs: notebook-3c11c2d5200e
